In [1]:
# ==========================================
# 📚 Urdu + Arabic PDF -> OCR (if needed) -> Clean -> Stem -> Chunk -> Embed -> FAISS
# ==========================================

import os, re, fitz, cv2, pytesseract
import pandas as pd
import numpy as np
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
from sentence_transformers import SentenceTransformer
import faiss
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from PIL import Image
import io

# Ensure stopwords downloaded
nltk.download('stopwords')

# --------------------------
# Step 1: Paths and Setup
# --------------------------
pdf_folder = "./sarf-books"
output_folder = "cleaned_sarf_books"
os.makedirs(output_folder, exist_ok=True)

urdu_stopwords = set(stopwords.words('arabic'))
custom_urdu = {'ہے','ہیں','تھا','تھی','کے','کی','کا','میں','سے','پر','اور','کو','نے','یہ','وہ','ہم','تم','کر','گیا','گئی'}
urdu_stopwords.update(custom_urdu)
arabic_stemmer = ISRIStemmer()

# 🔹 Tesseract path (update if different)
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# --------------------------
# Step 2: Helper Functions
# --------------------------
def clean_text(text):
    text = re.sub(r'[A-Za-z0-9]', '', text)
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return re.findall(r'[\u0600-\u06FF]+', text)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in urdu_stopwords]

def urdu_stemmer(word):
    suffixes = ['وں','یں','ات','اتوں','یاں','یوں','گی','گا','ی','ے','ا']
    for suf in suffixes:
        if word.endswith(suf) and len(word) > len(suf)+2:
            return word[:-len(suf)]
    return word

def stem_words(tokens):
    stemmed = []
    for t in tokens:
        try:
            s = arabic_stemmer.stem(t)
        except:
            s = urdu_stemmer(t)
        stemmed.append(s)
    return stemmed

# 🔹 Preprocess image before OCR
def preprocess_image(image_bytes):
    img = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(img, cv2.IMREAD_COLOR)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)   # increase contrast
    gray = cv2.GaussianBlur(gray, (3, 3), 0)  # denoise
    return gray

# 🔹 Extract text (with OCR fallback)
def extract_text_from_pdf(file_path, lang='urd+ara'):
    text = ""
    with fitz.open(file_path) as pdf:
        for page in pdf:
            page_text = page.get_text("text")
            if len(page_text.strip()) < 30:  # low text -> probably scanned
                img = page.get_pixmap(dpi=300)
                img_bytes = img.tobytes("png")
                gray = preprocess_image(img_bytes)
                ocr_text = pytesseract.image_to_string(gray, lang=lang)
                text += ocr_text + "\n"
            else:
                text += page_text + "\n"
    return text

# --------------------------
# Step 3: Process PDFs
# --------------------------
records = []
all_text = ""

pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]

for file_name in pdf_files:
    print(f"\n📘 Processing: {file_name}")
    file_path = os.path.join(pdf_folder, file_name)
    
    text = extract_text_from_pdf(file_path)
    cleaned = clean_text(text)
    tokens = tokenize(cleaned)
    filtered = remove_stopwords(tokens)
    stemmed = stem_words(filtered)
    final_text = " ".join(stemmed)
    
    out_path = os.path.join(output_folder, f"stemmed_{file_name.replace('.pdf','.txt')}")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(final_text)
    
    records.append({"book": file_name, "text": final_text})
    all_text += final_text + " "
    print(f"✅ Saved cleaned text for {file_name}")

rag_df = pd.DataFrame(records)
rag_df.to_csv("RAG_ready_books.csv", index=False, encoding="utf-8-sig")
print("\n✅ Saved combined dataset: RAG_ready_books.csv")

# --------------------------
# Step 4: Split Text into Chunks
# --------------------------
def split_into_chunks(text, max_words=200):
    words = text.split()
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

chunks = []
for idx, row in rag_df.iterrows():
    for chunk in split_into_chunks(row["text"], max_words=200):
        chunks.append({"book": row["book"], "chunk": chunk})

chunks_df = pd.DataFrame(chunks)
chunks_df.to_csv("RAG_chunks.csv", index=False, encoding="utf-8-sig")
print(f"✅ Created {len(chunks_df)} text chunks")

# --------------------------
# Step 5: Embeddings + FAISS
# --------------------------
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)

print("\n🔹 Generating embeddings (may take time)...")
embeddings = model.encode(chunks_df["chunk"].tolist(), show_progress_bar=True)
embeddings = np.array(embeddings, dtype="float32")

d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings)

faiss.write_index(index, "urdu_arabic_books.index")

print("\n✅ FAISS index created and saved as 'urdu_arabic_books.index'")
print(f"Total vectors stored: {index.ntotal}")

# --------------------------
# Step 6: Search Query
# --------------------------
def search_query(query, top_k=3):
    q_emb = model.encode([query])
    D, I = index.search(np.array(q_emb, dtype="float32"), top_k)
    results = []
    for idx in I[0]:
        if idx < len(chunks_df):
            results.append(chunks_df.iloc[idx]["chunk"])
    return results

query = "علم اور ایمان کا تعلق"
results = search_query(query, top_k=3)

print("\n🔍 Example RAG Search Results:")
for i, r in enumerate(results, 1):
    print(f"\nResult {i}:\n{r[:300]}...")


c:\Users\ASC\miniconda3\envs\myproject\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



📘 Processing: IRFAN-UL-SARF.pdf
✅ Saved cleaned text for IRFAN-UL-SARF.pdf

📘 Processing: IrshadUsSarfArabic.pdf
✅ Saved cleaned text for IrshadUsSarfArabic.pdf

📘 Processing: MINHAJ US SARF BY SADUL BAQI HAQQANI (1).pdf
✅ Saved cleaned text for MINHAJ US SARF BY SADUL BAQI HAQQANI (1).pdf

📘 Processing: nisab-us-sarf.pdf
✅ Saved cleaned text for nisab-us-sarf.pdf

📘 Processing: pdf   إيجاز التعريف في علم التصريف ، محمد بن مالك الطائي النحوي ، ت محمد عثمان.pdf
✅ Saved cleaned text for pdf   إيجاز التعريف في علم التصريف ، محمد بن مالك الطائي النحوي ، ت محمد عثمان.pdf

📘 Processing: TaleemUsSarf.pdf
✅ Saved cleaned text for TaleemUsSarf.pdf

📘 Processing: ارشاد_الصیغہ_شرح_اردو_علم_الصیغہ_مع_خاصیات_ابواب_.pdf
✅ Saved cleaned text for ارشاد_الصیغہ_شرح_اردو_علم_الصیغہ_مع_خاصیات_ابواب_.pdf

📘 Processing: تعلیلات خادمیہ.pdf
✅ Saved cleaned text for تعلیلات خادمیہ.pdf

📘 Processing: تیسیر ابواب الصرف.pdf
✅ Saved cleaned text for تیسیر ابواب الصرف.pdf

📘 Processing: دروس في علم الصرف.pdf
✅ Sav

c:\Users\ASC\miniconda3\envs\myproject\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



🔹 Generating embeddings (may take time)...


Batches: 100%|██████████| 41/41 [01:51<00:00,  2.72s/it]


✅ FAISS index created and saved as 'urdu_arabic_books.index'
Total vectors stored: 1285

🔍 Example RAG Search Results:

Result 1:
كر سن ےتقاعد لوہ اعد وی ےکہ اٹل لض فا كه يلت جور ضما علم بلك مف ہوگاور یسر الى نكل سط تام تہ وو جار علم شی نک سور ہوگ ددد بای جرد دع بد درا قرم قات شل ادل یا دہ حرف بال مول جد ہودوعی نل اع ركن دده قاعدہ طبن فعل مات ضار رع علم ترع ہے۔ اوررومرے تام اوب ضرر علم ام طو رہج ہتے۔ ےکسرہ كعت بعة ماس تکونایجت...

Result 2:
ترا ہا نک گب راک یکم ہو یقت بہ دبلل رف نیس تی رب بی کیاگیاکہ صرف علم دحو ابھ مال تی زلیس تکا سب ٹی ے می پھی وبا کیم ررف دا را ردل رین ہودومرکبعبارت ۱ پی ماس میس مو جووککم ماف جوا ہر عون ےگ؟ اعم صرف می زان نس يذ ھکی حیثی ركن صيفو پان ان شل جار قوا ین بيت حول صغو ںکی صل وغر علم بول ج١٠ ليل سم لصر مرف ...

Result 3:
رشل دآ ۓے ثيب زمٹہ زماشدوق توس ءا سبل مض اضى گذشنزائ کت ہیں۔ عال جدہز مان ۓےکوک ہیں۔ نیل ونے مان ۓےکوکتے ہیں۔ تح علم ول ےج فئل نلا زار گے صرزڈ زرۓ مدل ٹل بول ول ےجس فئل مت کرنے وال دده يج راڈ خالدک مسن ورک ول ٹس ےکسی ہون اکر مجھاجاۓے تج صر